In [0]:
from pyspark.sql.functions import *

In [0]:
dbutils.widgets.text('incremental_flag', '0')

In [0]:
incremental_flag = dbutils.widgets.get("incremental_flag")


In [0]:
%sql
select * from parquet.`abfss://silver@carsalesproject23.dfs.core.windows.net/carsales`


In [0]:
df_src = spark.sql('''SELECT DISTINCT Date_ID as date_id
FROM parquet.`abfss://silver@carsalesproject23.dfs.core.windows.net/carsales`''')

In [0]:
df_src.display()

In [0]:
if(spark.catalog.tableExists('cars_catalog.gold.dim_date')):
    df_sink = spark.sql('''select dim_date_key, date_id
                 FROM cars_catalog.gold.dim_date''')
else:
    df_sink = spark.sql('''select 1 as dim_date_key, 
                        Date_ID as date_id from 
                        parquet.`abfss://silver@carsalesproject23.dfs.core.windows.net/carsales`
                        where 1=0''')
df_sink.display()    

### Filtering old records and new records

In [0]:
df_filter = df_src.join(df_sink, df_src['date_id'] == df_sink['date_id'], 'left') \
               .select(df_src['date_id'],df_sink['dim_date_key'])   
df_filter.display()

In [0]:
df_old = df_filter.filter(col('dim_date_key').isNotNull())
df_old.display()
df_new = df_filter.filter(col('dim_date_key').isNull()).select('date_id')
df_new.display()

In [0]:
if incremental_flag == '0':
    max_key = 0
else :
    max_key = spark.sql('''select max(dim_date_key) from cars_catalog.gold.dim_date''').collect()[0][0]


In [0]:
df_filter_new = df_new.withColumn('dim_date_key', monotonically_increasing_id() + max_key + 1)
df_filter_new.display()


In [0]:
df_final = df_old.union(df_filter_new)
df_final.display()


### SCD Type 1

In [0]:
from delta.tables import DeltaTable

In [0]:
#incremnetal
if spark.catalog.tableExists('cars_catalog.gold.dim_date'):
    delta_tbl = DeltaTable.forPath(spark, 'abfss://gold@carsalesproject23.dfs.core.windows.net/dim_date')
    delta_tbl.alias('trg').merge(df_final.alias('src'), 'trg.dim_date_key = src.dim_date_key') \
                          .whenMatchedUpdateAll() \
                          .whenNotMatchedInsertAll() \
                          .execute()  
#initial
else:
    df_final.write.format('delta') \
            . mode('overwrite') \
            .option('path','abfss://gold@carsalesproject23.dfs.core.windows.net/dim_date') \
            .saveAsTable('cars_catalog.gold.dim_date')        

In [0]:
%sql
select * from cars_catalog.gold.dim_date